In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog_name", "", "catalog_name")
dbutils.widgets.text("config_schema_name", "", "config_schema_name")
dbutils.widgets.text("bz_schema_name", "", "bz_schema_name")

catalog_name = dbutils.widgets.get("catalog_name")
config_schema_name = dbutils.widgets.get("config_schema_name")
bz_schema_name = dbutils.widgets.get("bz_schema_name")

In [0]:
# Insert overwrite rows into the Delta table
spark.sql(f"""
    INSERT OVERWRITE {catalog_name}.{config_schema_name}.dqx_tables (table_id, table_name, schema_name, catalog_name)
    VALUES
        (1, 'customer', '{bz_schema_name}', '{catalog_name}'),
        (2, 'order', '{bz_schema_name}', '{catalog_name}')
""")

spark.sql(f"select * from {catalog_name}.{config_schema_name}.dqx_tables").display()

In [0]:
# Insert overwrite rows into the Delta table for dqx_columns
spark.sql(f"""
    INSERT OVERWRITE {catalog_name}.{config_schema_name}.dqx_columns (
        table_id, column_name, data_type, description, business_definition, metadata
    )
    SELECT 
        t.table_id,
        i.column_name,
        i.data_type,
        NULL AS description,
        NULL AS business_definition,
        NULL AS metadata
    FROM {catalog_name}.information_schema.columns i 
    JOIN {catalog_name}.{config_schema_name}.dqx_tables t 
      ON (lower(i.table_catalog) = lower(t.catalog_name)
        AND lower(i.table_name) = lower(t.table_name))
        AND lower(i.table_schema) = '{bz_schema_name}'
    WHERE column_name IN ('customer_id','customer_name','customer_dob','customer_email','customer_phone')
""")

spark.sql(f"select * from {catalog_name}.{config_schema_name}.dqx_columns").display()

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {catalog_name}.{config_schema_name}.dqx_rule_definitions 
    (rule_id, rule_name, rule_function, description, rule_type, rule_category, created_date, updated_date)
    VALUES 
    -- Null & Empty Checks
    ('R001', 'Not Null Check', 'is_not_null', 'Ensures the column contains no null values.', 'row_level', 'Null & Empty', current_timestamp(), current_timestamp()),
    ('R002', 'Not Empty Check', 'is_not_empty', 'Ensures string columns are not empty strings.', 'column_level', 'Null & Empty', current_timestamp(), current_timestamp()),
    -- Uniqueness & Identity
    ('R003', 'Unique Check', 'is_unique', 'Ensures all values in the column are distinct.', 'column_level', 'Uniqueness', current_timestamp(), current_timestamp()),
    ('R004', 'Primary Key Check', 'is_primary_key', 'Combines Not Null and Unique checks.', 'row_level', 'Uniqueness', current_timestamp(), current_timestamp()),
    -- Numeric & Range Checks
    ('R005', 'Greater Than Zero', 'is_greater_than', 'Ensures numeric values are positive.', 'column_level', 'Numeric & Range', current_timestamp(), current_timestamp()),
    ('R006', 'Range Check', 'is_between', 'Ensures value falls within a specific min/max.', 'column_level', 'Numeric & Range', current_timestamp(), current_timestamp()),
    -- String & Pattern Checks
    ('R007', 'Pattern Match', 'matches_regex', 'Validates column values against a specified regex pattern.', 'column_level', 'String & Pattern', current_timestamp(), current_timestamp()),
    ('R008', 'String Length', 'has_length_between', 'Checks if string length is within bounds.', 'column_level', 'String & Pattern', current_timestamp(), current_timestamp()),
    -- Date Checks
    ('R009', 'Date in Past', 'is_in_past', 'Ensures the date is not a future date.', 'column_level', 'Date', current_timestamp(), current_timestamp()),
    -- Aggregate / Table Level
    ('R010', 'Minimum Row Count', 'expect_table_row_count_to_be_between', 'Ensures table is not empty or meets a threshold.', 'aggregate', 'Aggregate', current_timestamp(), current_timestamp()),
    -- Foreign Key Check
    ('R011', 'Foreign Key Check', 'is_foreign_key', 'Ensures column values exist in a referenced table/column.', 'column_level', 'Foreign Key', current_timestamp(), current_timestamp())
""")

spark.sql(f"select * from {catalog_name}.{config_schema_name}.dqx_rule_definitions").display()

In [0]:
# Insert sample row into dqx_rule_mappings
spark.sql(f"""
    INSERT OVERWRITE {catalog_name}.{config_schema_name}.dqx_rule_mappings (
        table_id, rule_id, column_name, criticality, is_active, arguments, updated_at
    )
    VALUES 
    -- Customer
    ('1', 'R001', 'customer_id', 'warn', true, map('column', 'customer_id'), current_timestamp()),
    ('1', 'R003', 'customer_phone', 'error', true, map('columns', '["customer_phone"]'), current_timestamp())
    -- ('1', 'R003', 'customer_phone', 'error', true, null, current_timestamp()),
    -- Order
    -- ('2', 'R011', 'customer_id', 'error', true, map('columns', '["customer_phone"]'), current_timestamp())
""")

spark.sql(f"SELECT * FROM {catalog_name}.{config_schema_name}.dqx_rule_mappings").display()